# Init

In [17]:
# init
from importlib import reload
import os
from pathlib import Path
import pandas as pd

import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph
import toolsSync.main as tsm

def pckgs_reload():
    reload(tgm)
    reload(too)
    reload(tph)
    reload(tgl)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
TESTS_DIR = DATA_DIR / 'tests results'
CLEANED_DIR = DATA_DIR / 'cleaned'
TEST_BASIC_DIR = TESTS_DIR / 'osm basic test'
TEST_DUPLICATES_DIR = TESTS_DIR / 'osm duplicates test'
TEST_FIRST_LEVEL_DIR = TESTS_DIR / 'osm first level test'
FIXES_DIR = DATA_DIR / 'fixes'
FIXED_DIR = DATA_DIR / 'fixed'

logger = tgl.initiate_logger('logger', TEST_BASIC_DIR / 'basic_test.log')

process_state = tgm.load(DATA_DIR / "process_state.json")

In [11]:
df_by_cntr = tgm.load_dirs(CLEANED_DIR)
logger.info(f"Cleaned data for countries: {len(df_by_cntr)}")

[2026-01-26 14:44:17] [INFO] : Cleaned data for countries: 85


In [12]:
id_triplets = pd.concat(df_by_cntr.values(), ignore_index=True)[['id', 'tags.parent_id', 'tags.country_id']].fillna('missing').apply(tuple,axis=1).to_list()
logger.info(f"All dataframes id triplets: {len(id_triplets)}")

[2026-01-26 14:44:26] [INFO] : All dataframes id triplets: 197862


# basic test

### * review

In [13]:
test_res_by_cntr = tgm.load_dirs(TEST_BASIC_DIR)
logger.info(f'Test results found: {len(test_res_by_cntr)}')

[2026-01-26 14:45:12] [INFO] : Test results found: 85


In [14]:
temp = pd.DataFrame(test_res_by_cntr)
temp = temp.T.map(len)
temp['test_tags_total'] = [len(elems) for c,elems in df_by_cntr.items() if c in test_res_by_cntr.keys()]
temp.peek()

,missing_name,test_tags_leak,test_tags_in_country,test_tags_NA_result,test_tags_total
Afghanistan,0,0,24,11,35
Albania,0,0,390,0,390
Algeria,0,0,1,2147,2148
Andorra,0,0,1,0,1
Angola,0,0,1,63,64
Anguilla,0,0,1,0,1
AntiguaAndBarbuda,0,0,1,11,12
Argentina,1,0,885,326,1211
Armenia,0,0,27,0,27
Australia,0,0,1,615,616


### * select missing names and leaks from other countries

In [15]:
missing_names = set()
leaks = set()
for k,v in test_res_by_cntr.items():
    missing_names.update(v['missing_name'])
for k,v in test_res_by_cntr.items():
    leaks.update(v['test_tags_leak'])

logger.info(f"Missing names relations: {len(missing_names)}")
logger.info(f"Leaks relations: {len(leaks)}")
relations_from_test_to_delete = leaks | missing_names
logger.info(f"To delete relations (parents): {len(relations_from_test_to_delete)}")

[2026-01-26 14:45:26] [INFO] : Missing names relations: 24
[2026-01-26 14:45:26] [INFO] : Leaks relations: 27
[2026-01-26 14:45:26] [INFO] : To delete relations (parents): 51


### * from the relations to delete, select the childs in the country that has them as parent

In [16]:
parents_to_delete = relations_from_test_to_delete
relations_childs_to_delete = set()
while len(parents_to_delete) > 0:
    parents_id_and_countryid = {(ele[0],ele[2]) for ele in parents_to_delete}
    childs_to_delete = {ele for ele in id_triplets if (ele[1], ele[2]) in parents_id_and_countryid}
    relations_childs_to_delete.update(childs_to_delete)
    parents_to_delete = childs_to_delete
logger.info(f"Childs to delete: {len(relations_childs_to_delete)}")

[2026-01-26 14:45:30] [INFO] : Childs to delete: 4729


### * dump basic_test_to_delete.json

In [19]:
basic_test_to_delete = relations_from_test_to_delete | relations_childs_to_delete
logger.info(f"Basic test to delete relations: {len(basic_test_to_delete)}")

[2026-01-26 14:46:15] [INFO] : Basic test to delete relations: 4775


In [21]:
tgm.dump(FIXES_DIR / "basic_test_to_delete.pkl", basic_test_to_delete)

# first level test

### * review

In [22]:
first_level_test_res = tgm.load_dirs(TEST_FIRST_LEVEL_DIR)
logger.info(f'First level test results found: {len(first_level_test_res)}')

[2026-01-26 14:46:43] [INFO] : First level test results found: 68


In [23]:
first_level_test_res_by_elem = {k:v for list in first_level_test_res.values() for k,v in list.items()}
print(len(first_level_test_res_by_elem))

1270


In [24]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in first_level_test_res_by_elem.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                   777
[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]              482
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]      7
[[admin_centre, ok], [label, ok], [place, error], [geom_node, ok], [centroid, ok]]                  2
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                1
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, error], [centroid, ok]]                  1
Name: count, dtype: int64

In [72]:
less = set()
for elem, val in first_level_test_res_by_elem.items():
    if len(val) < 4: less.add(elem)
print(len(less))
{id_to_cntr[id[2]] for id in less}

0


set()

In [25]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         5843
missing     504
error         3
Name: count, dtype: int64

### * select parent entities to delete

In [26]:
MIN_TESTS = 2
KEEP_THRESHOLD = 0.3
miss = 0
first_level_test_res_bool = {}
for country, res in first_level_test_res.items():
    for id, val in res.items():
        true_count = 0
        false_count = 0
        for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
            if val[type]['status'] == 'ok':
                # make a voting weight selection
                if val[type]['result'] is True:
                    true_count += 1
                else:
                    false_count += 1
        if (true_count + false_count) < MIN_TESTS:
            first_level_test_res_bool[id] = 'missing'
        else:
            true_ratio = true_count / ((false_count + true_count))
            # test_res_by_tuple_bool[id] = true_count >= false_count
            first_level_test_res_bool[id] = true_ratio > KEEP_THRESHOLD

logger.info(f"Data types: {tgm.tally([v for k,v in first_level_test_res_bool.items()])}")

[2026-01-26 14:47:01] [INFO] : Data types: Counter({True: 1253, False: 17})


In [27]:
first_level_parents_to_delete = {k for k,v in first_level_test_res_bool.items() if v is False}
logger.info(f'Relations from test to delete: {len(first_level_parents_to_delete)}')

[2026-01-26 14:47:04] [INFO] : Relations from test to delete: 17


### * from the relations to delete, select the childs in the country that has them as parent

In [28]:
parents_to_delete = first_level_parents_to_delete
first_level_childs_to_delete = set()
while len(parents_to_delete) > 0:
    parents_id_and_countryid = {(ele[0],ele[2]) for ele in parents_to_delete}
    to_delete = {ele for ele in id_triplets if (ele[1], ele[2]) in parents_id_and_countryid}
    first_level_childs_to_delete.update(to_delete)
    parents_to_delete = to_delete
logger.info(f"Childs to delete: {len(first_level_childs_to_delete)}")

[2026-01-26 14:47:06] [INFO] : Childs to delete: 6318


In [29]:
first_level_to_delete = first_level_parents_to_delete | first_level_childs_to_delete
logger.info(f"Current relations to delete: {len(first_level_to_delete)}")

[2026-01-26 14:47:07] [INFO] : Current relations to delete: 6335


In [30]:
tgm.dump(FIXES_DIR  / "first_level_test_to_delete.pkl", first_level_to_delete)

# duplicates test

### * review

In [31]:
dups_test_res = tgm.load_dirs(TEST_DUPLICATES_DIR)
logger.info(f'Dups test results found: {len(dups_test_res)}')

[2026-01-26 14:47:34] [INFO] : Dups test results found: 34


In [32]:
dups_test_res_by_elem = {k:v for list in dups_test_res.values() for k,v in list.items()}
print(len(dups_test_res_by_elem))

2076


In [33]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in dups_test_res_by_elem.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                        1513
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]               270
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                         251
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                               34
[[admin_centre, missing], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                      6
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, missing], [centroid, missing]]       2
Name: count, dtype: int64

In [34]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         7784
missing    2596
Name: count, dtype: int64

### * select parent entities to delete

In [35]:
MIN_TESTS = 2
KEEP_THRESHOLD = 0.3
miss = 0
dups_test_res_bool = {}
for country, res in dups_test_res.items():
    for id, val in res.items():
        true_count = 0
        false_count = 0
        for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
            if val[type]['status'] == 'ok':
                # make a voting weight selection
                if val[type]['result'] is True:
                    true_count += 1
                else:
                    false_count += 1
        if (true_count + false_count) < MIN_TESTS:
            dups_test_res_bool[id] = 'missing'
        else:
            true_ratio = true_count / ((false_count + true_count))
            # test_res_by_tuple_bool[id] = true_count >= false_count
            dups_test_res_bool[id] = true_ratio > KEEP_THRESHOLD

logger.info(f"Data types: {tgm.tally([v for k,v in dups_test_res_bool.items()])}")

[2026-01-26 14:47:44] [INFO] : Data types: Counter({True: 1777, False: 297, 'missing': 2})


In [36]:
dups_parents_to_delete = {k for k,v in dups_test_res_bool.items() if v is False}
logger.info(f'Relations from test to delete: {len(dups_parents_to_delete)}')

[2026-01-26 14:47:47] [INFO] : Relations from test to delete: 297


### * from the relations to delete, select the childs in the country that has them as parent

In [37]:
parents_to_delete = dups_parents_to_delete
dups_childs_to_delete = set()
while len(parents_to_delete) > 0:
    parents_id_and_countryid = {(ele[0],ele[2]) for ele in parents_to_delete}
    to_delete = {ele for ele in id_triplets if (ele[1], ele[2]) in parents_id_and_countryid}
    dups_childs_to_delete.update(to_delete)
    parents_to_delete = to_delete
logger.info(f"Childs to delete: {len(dups_childs_to_delete)}")

[2026-01-26 14:47:50] [INFO] : Childs to delete: 1722


In [38]:
dups_to_delete = dups_parents_to_delete | dups_childs_to_delete
logger.info(f"Dups relations to delete: {len(dups_to_delete)}")

[2026-01-26 14:47:52] [INFO] : Dups relations to delete: 2010


In [39]:
tgm.dump(FIXES_DIR  / "dups_test_to_delete.pkl", dups_to_delete)

### * add manual review

In [40]:
to_review_triplets = [('72639','60189','60189'), ('72639','60199','60199'),('6543282', '6543265', '192756'),
 ('6543282', '6543273', '192756'), ('59190', '59195', '59065'), ('59190', '59752', '59065'), ('8309397', '391455', '148838'), ('8309397', '1116270', '148838')]

In [41]:
tgm.dump(FIXES_DIR / "to_review_triplets.pkl", to_review_triplets)

In [42]:
dups_manual_to_delete = {('6543282', '6543273', '192756'), ('59190', '59195', '59065')}

In [43]:
tgm.dump(FIXES_DIR / "dups_manual_to_delete.pkl", dups_manual_to_delete)

In [45]:
dups_final_to_delete = dups_to_delete | dups_manual_to_delete
logger.info(f"Dups joined relations to delete: {len(dups_final_to_delete)}")

[2026-01-26 14:50:04] [INFO] : Dups joined relations to delete: 2012


In [46]:
tgm.dump(FIXES_DIR / "dups_final_to_delete.pkl", dups_final_to_delete)

# apply fixes

### * countries without all test completed

In [47]:
wihout_completed_tests = set()
for country in df_by_cntr.keys():
    state = process_state[country]
    for task in ['test_basic', 'test_first_level', 'test_duplicates']:
        if state[task]['status'] not in ['ok', 'missing']:
            wihout_completed_tests.add(country)
logger.info(len(wihout_completed_tests))

[2026-01-26 14:50:23] [INFO] : 0


In [48]:
wihout_completed_tests

set()

### * join relations to delete

In [51]:
basic_to_delete = tgm.load(FIXES_DIR / "basic_test_to_delete.pkl")
logger.info(len(basic_to_delete))

first_level_to_delete = tgm.load(FIXES_DIR / "first_level_test_to_delete.pkl")
logger.info(len(first_level_to_delete))

dups_joined_to_delete = tgm.load(FIXES_DIR / "dups_final_to_delete.pkl")
logger.info(len(dups_joined_to_delete))

[2026-01-26 14:52:29] [INFO] : 4775
[2026-01-26 14:52:29] [INFO] : 6335
[2026-01-26 14:52:29] [INFO] : 2012


In [52]:
to_delete = basic_to_delete | first_level_to_delete | dups_joined_to_delete
print(len(to_delete))

7052


In [53]:
df_by_cntr_fixed = {}
for k,df in df_by_cntr.items():
    df_by_cntr_fixed[k] = df[~df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(to_delete)]
print(len(df_by_cntr_fixed))

85


In [55]:
id_triplets_new = pd.concat(df_by_cntr_fixed.values(), ignore_index=True)[['id', 'tags.parent_id', 'tags.country_id']].fillna('missing').apply(tuple,axis=1).to_list()
logger.info(f"All dataframes id triplets: {len(id_triplets_new)}")

[2026-01-26 14:52:52] [INFO] : All dataframes id triplets: 190810


In [56]:
print(len(id_triplets))
print(len(id_triplets)-len(to_delete))
print(len(id_triplets_new))

197862
190810
190810


### * save

In [57]:
for country, df in df_by_cntr_fixed.items():
    tgm.dump(FIXED_DIR / country / f"{country}_fixed.pkl", df)